In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

import plotly.io as pio

pio.templates.default = "plotly_white"

In [ ]:
template_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
pdb_ids = template_df.index.to_list()

In [ ]:
ensembles_dir = "../data/mhcii_tcr_ensembles/"
sampling_methods = ["static/crystal", "static/pandora2", "ensemble/annealing", "ensemble/pandora2"]
docking_method = "haddock3_rigidbody"

In [ ]:
df = None

for pdb_id in pdb_ids:
    for sampling_method in sampling_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method, docking_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["sampling_method"] = sampling_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

df

In [ ]:
metrics = ["rmsd", "dockq", "dockq_fnat", "binding_core_sasa"]

In [ ]:
pdb_ids = template_df.index.to_list()
colors = ["blue", "limegreen", "crimson", "green"]

# centroid = template_df[["crossing_angle", "incident_angle"]].values.mean(axis=0)
# template_df["eccentricity"] = np.linalg.norm(template_df[["crossing_angle", "incident_angle"]].values - centroid, axis=1)
# px.line(template_df, x=template_df.index.to_list(), y="eccentricity").show()

In [ ]:
from plotly.subplots import make_subplots

for metric in metrics:
    # fig = px.strip(df, x="pdb_id", y=metric, color="sampling_method")
    # fig.show()
    fig = go.Figure()
    for i, pdb_id in enumerate(pdb_ids):
        pdb_df = df[df.pdb_id == pdb_id]
        for sampling_method, color in zip(sampling_methods, colors):
            ens_df = pdb_df[pdb_df.sampling_method == sampling_method]
            fig.add_trace(
                go.Box(
                    y=ens_df[metric].values,
                    name=pdb_id,
                    offsetgroup=sampling_method,
                    legendgroup=sampling_method,
                    showlegend=(i == 0),
                    line=dict(color=color),
                    marker=dict(color=color)
                )
            )
    fig.update_xaxes(showticklabels=True)
    fig.update_layout(title=metric, boxmode="group")
    fig.show()

In [ ]:
colors = ['maroon', 'orange', 'purple', 'green']

for sampling_method in sampling_methods:
    sub_df = df[df.sampling_method == sampling_method]
    bins = [0, 0.24, 0.51, 0.81, 1.0]
    labels = ['<0.24', '0.24–0.51', '0.51–0.81', '>0.81']
    sub_df['dockq_bin'] = pd.cut(sub_df['dockq'], bins=bins, labels=labels, include_lowest=True)
    counts = sub_df.groupby(['pdb_id', 'dockq_bin']).size().unstack(fill_value=0)
    proportions = counts.div(counts.sum(axis=1), axis=0)
    proportions = proportions[labels]
    fig = go.Figure()
    x_vals = proportions.index.tolist()
    for i, col in enumerate(proportions.columns):
        fig.add_bar(
            x=x_vals,
            y=proportions[col],
            name=col,
            marker_color=colors[i]
        )
    fig.update_layout(
        title=sampling_method,
        barmode='stack',
        xaxis_title='pdb_id',
        yaxis_title='Proportion',
        yaxis=dict(range=[0, 1])
    )
    fig.show()

In [ ]:
for metric in metrics:
    for sampling_method in ["ensemble/annealing"]:
        fig = px.scatter(df[df.sampling_method == sampling_method], x="crossing_angle", y="incident_angle", color=metric, title=sampling_method)
        fig.update_xaxes(range=(0, 180))
        fig.update_yaxes(range=(0, 90))
        fig.show()

In [ ]:
df["neg_bcs"] = df.binding_core_sasa * -1

for sampling_method in ["ensemble/annealing"]:
    fig = px.scatter_3d(df[df.sampling_method == sampling_method], x="crossing_angle", y="incident_angle", z="dockq", color="binding_core_sasa", title=sampling_method)
    fig.update_xaxes(range=(0, 180))
    fig.update_yaxes(range=(0, 90))
    # fig.update_layout(scene=dict(zaxis=dict(range=(0, 1))))
    fig.show()

In [ ]:
from scipy.stats import spearmanr
from sklearn.cross_decomposition import PLSRegression

for sampling_method in ["ensemble/pandora2"]:

    sub_df = df[df.sampling_method == sampling_method]
    print(len(sub_df))

    X = sub_df[["crossing_angle", "incident_angle", "binding_core_sasa"]].values
    y = sub_df["dockq"].values

    pls = PLSRegression(n_components=2)
    pls.fit(X, y)

    y_hat = pls.predict(X)

    fig = px.scatter(x=y, y=y_hat, title=sampling_method, hover_name=sub_df.index.to_list())
    fig.update_xaxes(range=(0, 1))
    fig.show()
    print(spearmanr(y, y_hat))

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor

selections = pd.DataFrame(columns=df.columns)

for sampling_method in ["ensemble/annealing"]:

    max_dockqs = list()
    spearmans = list()
    pdb_ids = list()

    sub_df = df[df.sampling_method == sampling_method]

    for pdb_id in sub_df.pdb_id.unique():

        print(pdb_id)

        pdb_ids.append(pdb_id)

        train_df = sub_df[sub_df.pdb_id != pdb_id]
        test_df = sub_df[sub_df.pdb_id == pdb_id]

        X_train = train_df[["crossing_angle", "incident_angle", "binding_core_sasa"]].values
        y_train = train_df["dockq"].values

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        X_test = test_df[["crossing_angle", "incident_angle", "binding_core_sasa"]].values
        y_test = test_df["dockq"].values
        max_dockqs.append(y_test.max())
        y_hat = model.predict(X_test)
        test_df["y_hat"] = y_hat
        selections = pd.concat([selections, test_df.sort_values(by="y_hat", ascending=False)])

        spearmans.append(spearmanr(y_test, y_hat)[0])

    fig = px.box(x=spearmans, title=sampling_method, hover_name=pdb_ids)
    fig.update_xaxes(range=(0, 1))
    fig.update_yaxes(range=(-1, 1))
    fig.show()

In [ ]:
import plotly.express as px

bins = [0, 0.24, 0.51, 0.81, 1.0]

def foo(x):

    for i in range(5):
        if bins[i] > x:
            return i - 1

matrix_1 = np.zeros((6, len(pdb_ids)))
matrix_2 = np.zeros((6, len(pdb_ids)))

for i, pdb_id in enumerate(pdb_ids):
    sub_df = selections[selections.pdb_id == pdb_id].sort_values(by="y_hat", ascending=False)
    ranks = sub_df.dockq.rank(ascending=False).astype(int)
    # x.append(sub_df.iloc[0].dockq)
    # y.append(sub_df.iloc[0].y_hat)
    print(ranks.loc[sub_df.y_hat.idxmax()])
    for j, k in enumerate([1, 5, 10, 20, 50, 100]):
        matrix_1[j, i] = foo(sub_df.head(k).dockq.max())
        matrix_2[j, i] = ranks.head(k).min()

algae = px.colors.sequential.algae

eps = 1e-6
custom_scale = [[0.0, "white"]]

n = len(algae)
for i, color in enumerate(algae):
    t = eps + (1 - eps) * (i / (n - 1))
    custom_scale.append([t, color])

fig = go.Figure(data=go.Heatmap(z=matrix_1, x=pdb_ids, y=[f"top {i}" for i in [1, 5, 10, 20, 50, 100]], colorscale=custom_scale, showscale=False, xgap=1, ygap=1))
fig.update_layout(yaxis_scaleanchor="x")
fig.update_yaxes(domain=[0, 1])
fig.show()

fig = go.Figure(data=go.Heatmap(z=matrix_2, x=pdb_ids, y=[f"top {i}" for i in [1, 5, 10, 20, 50, 100]], colorscale="purp", reversescale=True, xgap=1, ygap=1))
fig.update_layout(yaxis_scaleanchor="x")
fig.update_yaxes(domain=[0, 1])
fig.show()

print(matrix_2[0, :].mean())
print(matrix_2[1, :].mean())
print(matrix_2[2, :].mean())
print(matrix_2[3, :].mean())
print(matrix_2[4, :].mean())
print(matrix_2[5, :].mean())
print((matrix_2 == 1).mean(1))